# VNC Desktop Demo

This notebook starts a virtual desktop inside the Binder container and opens it in your browser via noVNC.

Architecture:
```
Xvfb (virtual display) → x11vnc (VNC server) → websockify (WebSocket bridge) → browser via noVNC
```

Run the cells in order.

In [ ]:
# Cell 1: Start the VNC stack
import subprocess, os, time
from IPython.display import HTML, display

DISPLAY_NUM = 20
VNC_PORT    = 5920
NOVNC_PORT  = 6920

# Clean up any previous run
os.system(f"pkill -f 'Xvfb :{DISPLAY_NUM}' 2>/dev/null; "
          f"pkill -f 'x11vnc.*{VNC_PORT}' 2>/dev/null; "
          f"pkill -f 'websockify.*{NOVNC_PORT}' 2>/dev/null")
time.sleep(1)

env = {**os.environ, "DISPLAY": f":{DISPLAY_NUM}"}

# Start virtual framebuffer
subprocess.Popen(["Xvfb", f":{DISPLAY_NUM}", "-screen", "0", "1280x1024x24", "-ac"])
time.sleep(1)

# Start VNC server reading from Xvfb
subprocess.Popen([
    "x11vnc", "-display", f":{DISPLAY_NUM}",
    "-rfbport", str(VNC_PORT), "-forever", "-nopw", "-quiet"
])
time.sleep(1)

# Start websockify to bridge VNC -> WebSocket for noVNC
subprocess.Popen([
    "websockify", "--web", "/usr/share/novnc/",
    str(NOVNC_PORT), f"localhost:{VNC_PORT}"
])
time.sleep(1)

# Build URL — works both in JupyterHub and standalone Jupyter
base = os.environ.get("JUPYTERHUB_SERVICE_PREFIX", "/")
url  = f"{base}proxy/{NOVNC_PORT}/vnc.html?autoconnect=true&resize=scale"

print("VNC stack started")
display(HTML(f'<a href="{url}" target="_blank" style="font-size:1.2em">&#128444; Open VNC Desktop</a>'))

## Launch a graphical application

Run either cell below — the window will appear in the VNC desktop tab.

In [ ]:
# xeyes — classic X11 demo app (just to prove the display works)
import subprocess, os
env = {**os.environ, "DISPLAY": ":20"}
subprocess.Popen(["xeyes"], env=env)
print("xeyes launched — switch to the VNC desktop tab")

In [ ]:
# matplotlib window via Tk backend — closer to real physics use
import subprocess, os
script = """
import matplotlib
matplotlib.use("TkAgg")
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(7, 5))
theta = np.linspace(0, 2*np.pi, 500)
for r in [1, 2, 3]:
    ax.plot(r*np.cos(theta), r*np.sin(theta), label=f"track r={r}")
ax.set_aspect("equal")
ax.set_title("Toy event display (matplotlib/Tk)")
ax.legend()
plt.show()
"""
env = {**os.environ, "DISPLAY": ":20"}
subprocess.Popen(["python3", "-c", script], env=env)
print("matplotlib window launched — switch to the VNC desktop tab")

In [ ]:
# Cleanup — stop the VNC stack
import os
os.system("pkill xeyes; pkill x11vnc; pkill websockify; pkill Xvfb")
print("VNC stack stopped")